# Scala Basics for Data Engineers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/scala.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is Scala and why do data engineers learn it?

Scala is a statically typed JVM language that blends object-oriented and functional programming. Spark is written in Scala, the official Kafka clients are Java, and Flink exposes a JVM DataStream API, so Scala sits naturally next to all three.

```text
Kafka (Java clients) -> Spark/Flink JVM job -> lakehouse tables -> BI / ML
```

Data engineers learn Scala because:

+ Spark's API and internals are Scala-first, so source code and error messages are easier to read
+ Typed `Dataset[T]` code catches schema mistakes at compile time instead of at runtime
+ JVM-native logic avoids Python UDF serialization overhead for CPU-bound transforms
+ One library can be shared by Spark jobs, Flink jobs, and microservices

Real-world example: a team keeps revenue logic in one Scala library, and both a Spark batch job and a Flink streaming job call the same `OrderTransforms` object, so batch and streaming cannot silently diverge.

## How do vals and vars differ, and why does immutability matter?

`val` binds a name once; `var` can be reassigned. Default to `val` and use `var` only for tight local accumulation. `val` makes the reference immutable, not the object, so a `val` can still point at a mutable collection.

```scala
val rawPath = "s3://acme-raw/orders/"   // immutable binding
var attempt = 0                          // reassignable

This matters in Spark and Flink because closures are serialized and shipped to executors or task managers. A captured mutable variable is copied, so updates on executors are invisible on the driver. Immutable values remove that failure mode and are safe to share across threads and Futures.

## What are Scala basic types and how does type inference work?

Scala has the JVM primitives (`Int`, `Long`, `Double`, `Boolean`, `Char`) plus `String`, `BigInt`, and `BigDecimal`. Everything is an object at the API level, with `Any` at the top, `AnyVal`/`AnyRef` below it, and `Nothing` at the bottom. The compiler infers types, so annotate when widening or when you want a stable contract.

```scala
val rows = 1_000_000L             // Long
val ratio = 0.42                  // Double
val ids = Vector(1L, 2L, 3L)      // Vector[Long]
val widened: Long = rows + 1      // explicit widening
val total: BigDecimal = BigDecimal("19.99") * 3
```

Two habits help in interviews:

+ Annotate public return types and dataset schemas; inference is for locals
+ `Nothing` is a subtype of everything, which is why `if (x) 1 else throw new RuntimeException` still infers `Int`

For money prefer `BigDecimal` over `Double`: Spark maps `decimal(p, s)` to `BigDecimal`, and floating-point rounding in revenue numbers is a classic production incident.

## How does string interpolation work in Scala?

Three built-in interpolators: `s` substitutes values, `f` formats them, and `raw` disables escape processing. Interpolation accepts expressions in braces, so paths and log lines read cleanly.

```scala
val table = "silver.orders"
val rows = 4200L
val ratio = 0.127
val dt = "2026-10-04"

`f` type-checks the format specifier at compile time. `s` calls `toString`, so an `Option` prints as `Some(...)`; use `.getOrElse("")` before interpolating it into a message.

## How does control flow work in Scala with if/else and match?

`if`/`else` is an expression that returns a value, so it can initialize a `val`. `match` is the general branch construct and replaces long `if`/`else if` chains and `switch`, including guards such as `case Order(_, _, amount, "PAID") if amount > 100`.

```scala
val amount = BigDecimal("1500.00")
val tier = if (amount > 1000) "high" else "standard"

Because `OrderEvent` is `sealed`, the compiler warns if a case is missing. That exhaustiveness check is the main reason to model events as sealed hierarchies instead of strings.

## How do you define functions and methods in Scala?

`def` defines a method and can also define a local function inside another `def`. Scala supports default arguments, named arguments, and varargs written `Int*`.

```scala
def normalize(name: String, trim: Boolean = true, lower: Boolean = true): String = {
  var out = name
  if (trim) out = out.trim
  if (lower) out = out.toLowerCase
  out
}

A method returning `Unit` is written `def commit(): Unit = ...`. The older procedure syntax (`def commit() { ... }`) was deprecated in Scala 2.13 and removed in Scala 3, so an interviewer may check whether you still write it.

## What are higher-order functions and function values?

Functions are values. `A => B` is a type, and a method that takes or returns a function is higher-order. This is the basis of `map`, `filter`, `fold`, and Spark's typed transformations.

```scala
val double: Int => Int = _ * 2
val amounts = List(10, 20, 30)

In Spark, `_.amount` compiles to bytecode instead of a pickled Python lambda, which removes a serialization boundary between the driver and the executors.

## What is currying and partial application?

Currying splits a multi-argument function into a chain of single-argument functions, written as multiple parameter lists. The left lists become reusable configuration and the right list stays the changing data.

```scala
def add(a: Int)(b: Int): Int = a + b
val addTen = add(10)(_)           // Scala 2.13 placeholder; in Scala 3 write add(10)
addTen(5)                         // 15

Currying is not the same as default arguments: defaults fill in values at the call site, while currying returns a reusable function value after the first list is applied.

## How do case classes and pattern matching work?

A `case class` gets compiler-generated `equals`, `hashCode`, `toString`, `copy`, and companion `apply`/`unapply`, which makes it the standard immutable record type for domain models and Spark encoders.

```scala
final case class Order(orderId: Long, customerId: Long, amount: BigDecimal, status: String)

`copy` keeps record updates explicit, which matches Spark's immutable Dataset model. Field order and types must match the source: reading CSV straight into a case class with a `BigDecimal` field fails if the underlying column is a string, so read as a `DataFrame`, cast, then call `.as[Order]`.

## What are sealed traits and extractors?

A `sealed trait` restricts implementations to the same file, so the compiler can prove that a `match` covers every case. That is exactly what you want for event types and job states.

```scala
sealed trait OrderEvent
final case class Created(orderId: Long, customerId: Long) extends OrderEvent
final case class Updated(orderId: Long, amount: BigDecimal) extends OrderEvent
final case class Deleted(orderId: Long) extends OrderEvent
case object RolledBack extends OrderEvent

Real-world example: an ingestion job matches `OrderEvent` to route records to insert, merge, or delete paths, and adding a new event type makes the compiler report a non-exhaustive match at every routing site (an error under fatal warnings).

## How do traits, mixins, and abstract classes work?

Traits define interfaces plus optional concrete behavior and are mixed in with `with`. A class can extend one class but mix in many traits, which is how data code composes concerns like logging, metrics, and clocks.

```scala
trait Clock { def now(): Long }
trait Logging { def log(message: String): Unit = println(s"[${getClass.getSimpleName}] $message") }
abstract class BaseJob(val name: String) { def run(): Unit }

An abstract class is the better choice when you need constructor parameters or Java interoperability; traits are better for stackable behavior. Traits are linearized left to right, so the rightmost mixin wins for conflicting concrete methods, which is why `abstract override` is needed to wrap `super.run()`.

## What are objects and companion objects with apply and unapply?

An `object` is a singleton. An object with the same name as a class or case class in the same file is that class's companion, and it can see the class's private members. Companions are the idiomatic home for factories, constants, and `unapply`.

```scala
final case class Partition(dt: String, region: String)

`apply` is syntax sugar for calling the object (`obj(args)` means `obj.apply(args)`), and `unapply` is what pattern matching calls to take a value apart. Defining both on a companion gives a plain class case-class-like ergonomics.

## How do Scala collections and core combinators work?

`List` is a linked list with cheap head/tail, `Vector` is the indexed immutable sequence, `Seq` is the general sequence abstraction, and `Map`/`Set` are the associative and unique collections. In Scala 2.13 the default `Seq` is `scala.collection.immutable.Seq`.

```scala
val statuses: List[String] = List("PAID", "PENDING", "PAID")
val regions: Set[String] = Set("us", "eu", "us")

`fold`/`foldLeft` carries an initial accumulator and can change the result type; `reduce` seeds from the first element and fails on empty input, which is why `reduceOption` exists. Use `.view.mapValues(...)` to transform a `Map` lazily instead of rebuilding it eagerly.

## How do for-comprehensions desugar in Scala?

A `for` expression is syntax sugar: generators become `flatMap`, the final `yield` becomes `map`, filters become `withFilter`, and a `for` without `yield` becomes `foreach`.

```scala
val pairs = for {
  a <- List(1, 2, 3)
  b <- List(10, 20) if a < 3
} yield (a, b)

Because the desugaring only needs those methods, the same syntax sequences `Option`, `Either`, and `Future`:

```scala
def lookupCustomer(id: Long): Option[String] = if (id == 10L) Some("alice") else None

Repository-style lookups return `Option`, so a chain of enrichment steps reads top to bottom and short-circuits on the first missing value instead of nesting `if` blocks or throwing.

## How do you use Option, Either, and Try instead of null?

Scala encodes failure in the type system. `Option[A]` means "a value or nothing", `Either[L, R]` carries a typed error on the left, and `Try[A]` captures a thrown exception.

```scala
import scala.util.{Failure, Success, Try}

Why this matters at data boundaries:

+ Java APIs, JDBC, and environment variables return `null`; wrap them immediately with `Option(...)`
+ Spark `Dataset[T]` rows can still contain `null` in a case class field, so validate with a filter before writing a governed table

## What are implicits, givens, and extension methods?

Implicits are Scala 2's mechanism for automatically passing values and applying conversions. Scala 3 renamed the concept to `given`/`using` and added `extension` methods. Both add behavior to types you do not own, most commonly Spark's `Dataset`.

Scala 2.13 form, which is what Spark 3.x and Spark 4.x modules compile against:

```scala
import org.apache.spark.sql.Dataset

Scala 3 form of the same idea, plus an implicit type class instance:

```scala
// Scala 3 only: Spark core artifacts are published for Scala 2.12/2.13, not for Scala 3
extension (ds: Dataset[Order])
  def logCount(): Dataset[Order] = { println(s"rows=${ds.count()}"); ds }

Be honest about versions: `implicit` is the safe answer for Spark work today, while `given`/`extension` need a Scala 3 toolchain. Keep implicits narrow and local, because broad implicit conversions slow compilation and make type errors hard to read.

## How do generics and type bounds work in Scala?

Generic methods and classes use type parameters in square brackets, like Java generics but with declaration-site variance and richer bounds.

```scala
trait Source[+A] { def read(): A }          // covariant: produces A
trait Sink[-A]   { def write(a: A): Unit }  // contravariant: consumes A

Bounds that show up most in data code:

+ `<: A` upper bound requires a subtype, for example `Serializable` before shipping data across the network
+ `: Ordering` context bound supplies an implicit `Ordering[A]` so `sortBy` and `max` work
+ `: ClassTag` context bound supplies runtime class information, needed for arrays or encoders in generic code

## What are lazy val and by-name parameters?

A `lazy val` is initialized on first access and memoized. A by-name parameter (`=> T`) defers evaluation until the method body uses it, and re-evaluates on each use unless you cache it.

```scala
lazy val spark = SparkSession.builder().appName("orders").getOrCreate()

By-name parameters power assertions, retries, and cheap logging. In Spark jobs, `lazy val` for the `SparkSession` and for shared schemas avoids creating sessions during unit tests that never touch Spark. `lazy val` is thread-safe by default, so concurrent first access blocks until initialization completes.

## How do Futures and basic concurrency work in Scala?

`Future[T]` is a value that will be available later. Computation starts when the `Future` is created, runs on an `ExecutionContext`, and is composed with `map`, `flatMap`, and `Future.sequence`.

```scala
import scala.concurrent.{Await, ExecutionContext, Future}
import scala.concurrent.duration._

Rules that matter in data pipelines:

+ Do not use the global execution context for blocking work such as JDBC or file IO; size a dedicated pool, and bound fan-out because hundreds of concurrent Spark job submissions from one driver is worse than batching

## How does Java interoperability work in Scala?

Scala runs on the JVM, calls Java directly, and extends Java classes. The friction is mostly collection conversions and `null`, which is why `scala.jdk.CollectionConverters` exists.

```scala
import java.time.Instant
import java.util.{List => JList, Properties}
import scala.jdk.CollectionConverters._

Interop points interviewers check:

+ `scala.jdk.CollectionConverters._` is current in 2.13 and 3; the older `JavaConverters` is deprecated
+ Java methods that can return `null` should be wrapped with `Option(...)` at the call site
+ Scala annotations such as `@BeanProperty` and `@throws` expose Java-visible members when Java must consume the class
+ Java builders and setters stay as they are, which is why Kafka configuration uses `props.put(...)` rather than Scala collections

## How do you structure an sbt project with dependencies and tests?

sbt is the standard Scala build tool. A job project has a small, predictable layout, and the build file is itself Scala code.

```text
my-job/
  build.sbt
  src/main/scala/com/acme/OrdersJob.scala
  src/test/scala/com/acme/OrdersJobSpec.scala
```

```scala
// build.sbt
ThisBuild / scalaVersion := "2.13.14"

`%%` appends the Scala binary version (resolving `spark-sql_2.13`), and Spark dependencies are `Provided` because the cluster already ships them. `Test / fork := true` gives tests their own JVM so Spark and sbt classloaders do not conflict. Tests use MUnit below; ScalaTest works the same way with `AnyFunSuite` and `assert`.

```scala
// src/test/scala/com/acme/OrdersJobSpec.scala
import munit.FunSuite

Build and test with `sbt compile`, `sbt test`, `sbt "runMain com.acme.OrdersJob --dt 2026-10-04"`, or `sbt publishLocal` to share the transforms library with other jobs.

## What Scala idioms matter for Spark?

Spark's Scala API exposes typed `Dataset[T]`, `Encoders`, strongly typed aggregations, and direct access to internals. The main design decision is typed versus untyped.

```scala
import org.apache.spark.sql.{Dataset, SparkSession}
import org.apache.spark.sql.functions.{broadcast, col, udf}

Typed versus untyped:

+ `Dataset[T]` with case classes gives compile-time field checks and `Encoder[T]` derivation via `import spark.implicits._`
+ `DataFrame` (`Dataset[Row]`) with `Column` handles runtime-only schemas, is usually better optimized because Catalyst sees the whole expression tree, and suits wide or dynamic data

When Scala beats PySpark: CPU-heavy row-wise logic, logic shared as a JVM library with Kafka or Flink, and teams that want schema errors at build time. PySpark still wins on notebook iteration, the Python ML ecosystem, hiring friction, and built-in transformations where the JVM difference is small. Use a UDF only when no built-in function exists, and broadcast the small side of a join.

## What are common Scala pitfalls in data engineering?

+ `null` leaks from Java and from Spark rows. Wrap with `Option(...)` at the boundary, and remember that a `null` inside a `Dataset[T]` field does not throw until you call a method on it
+ `==` versus `equals`: `==` calls `equals` and is null-safe, so `a == b` is usually right. It is structural for case classes but reference-based for `Array`, where `sameElements` is what you want; `eq` is reference identity
+ Mutable collections captured in closures are copied to executors, so mutations are lost. Keep transformations pure and return values
+ Implicit conversions hide real errors and slow compilation; Scala 3 requires `import scala.language.implicitConversions` to use them, which is a hint to avoid them
+ Read stack traces bottom-up. Start from the deepest `Caused by` and the first frame in your own package; Spark wraps failures in `SparkException`, so the useful line is usually several causes down
+ Version mismatch is a real production failure: Spark 3.x artifacts exist for Scala 2.12 and 2.13, while Spark 4.x dropped 2.12 and publishes 2.13 only; adding a Scala 3 or mismatched 2.13 library to a 2.12 cluster gives `NoSuchMethodError` at runtime
+ Scala 2 versus Scala 3: `implicit`/`implicit class` versus `given`/`using`/`extension`; `_` versus `?` for wildcard types; procedure syntax and symbol literals removed; in 2.13 `Seq` defaults to `scala.collection.immutable.Seq`

A defensive pattern for the `null` pitfall:

```scala
def validate(o: RawOrder): Either[String, RawOrder] = {
  if (o.status == null || o.status.trim.isEmpty) Left(s"missing status for order ${o.orderId}")
  else if (o.amount == null || o.amount <= 0)    Left(s"bad amount for order ${o.orderId}")
  else Right(o.copy(status = o.status.trim.toUpperCase))
}
```

Route `Left` values to a quarantine table and continue, rather than throwing and failing the whole partition.

## Real world use cases for Scala in data engineering

### 1. Typed Spark Dataset pipeline

Case classes plus `Dataset[T]` make the transform contract explicit, and `partitionBy` keeps output rewritable per day.

```scala
import org.apache.spark.sql.{Dataset, SparkSession}

Why Scala: a renamed or retyped case class field fails the build instead of the midnight run, and `transform` is unit-testable on a local `SparkSession` with a small `Seq` input.

### 2. Kafka consumer that lands raw events

The official Kafka client is Java, so Scala consumes it directly and pattern matches on the decoded payload for routing.

```scala
import java.time.Duration
import java.util.{Collections, Properties}
import org.apache.kafka.clients.consumer.{ConsumerConfig, KafkaConsumer}
import org.apache.kafka.common.serialization.StringDeserializer
import scala.jdk.CollectionConverters._

Why Scala: disabling auto-commit and committing after the batch is durable gives at-least-once delivery with a controlled checkpoint, and the consumer can ship in the same JVM artifact as the transformation library.

### 3. Flink job sketch

Flink exposes a JVM DataStream API with a Scala layer, so a Scala service can own a streaming job end to end.

```scala
import org.apache.flink.api.common.eventtime.WatermarkStrategy
import org.apache.flink.api.common.serialization.SimpleStringSchema
import org.apache.flink.connector.kafka.source.KafkaSource
import org.apache.flink.connector.kafka.source.enumerator.initializer.OffsetsInitializer
import org.apache.flink.streaming.api.scala._

Why Scala: the job is a normal JVM deployment with the same build, logging, and tests as the rest of the platform. Version caveat: Flink's Scala DataStream API has been deprecated and removed in recent releases, so prefer the Java API for new projects.

### 4. Shared transformation library for batch and streaming

One library keeps batch and streaming definitions from drifting apart.

```scala
// src/main/scala/com/acme/transforms/OrderTransforms.scala
package com.acme.transforms

Published with `sbt publishLocal` or to the company artifact repository, then pulled into the batch job and the streaming job as one versioned dependency, so a revenue-rule change is reviewed and released once.

### 5. Streaming ETL with Futures and bounded batching

A micro-batch streaming ETL step consumes a checkpointed stream, and catch-up or replay work is driven with `Future` batching so the driver stays in control of write parallelism.

```scala
import scala.concurrent.{Await, Future}
import scala.concurrent.ExecutionContext.Implicits.global
import scala.concurrent.duration._

Why Scala: bounded batches stop a driver from submitting thousands of concurrent jobs while draining a backlog, completion is tracked per batch so a failed group can be retried, and the row counts feed data-quality checkpoints.